# Studi Kasus 2: Dual-Stage Early Warning System (EWS) Deteksi Dini Drop-Out (Mode Lokal)

Notebook ini merupakan versi rilis publik **100% lokal & teranonimisasi** untuk pilar **Student Retention / Drop-Out EWS**. Seluruh kolom PII telah dihapus dan wilayah geografis telah disamarkan menjadi `Zone A`–`Zone F` / `Other Zones`.

**Fokus Analitik (DS / ML / DA / BI):**
1. Memuat `../data/dropout_anonymized.csv` dan `../data/dropout_reasons_anonymized.csv` secara lokal.
2. Analisis diagnostik kesenjangan kehadiran, risiko per program, dan Pareto alasan keluar.
3. Replikasi evaluasi model Dual-Stage (`LogisticRegression` threshold `0.25` & `GradientBoostingClassifier` threshold `0.50`).
4. Uji inferensi end-to-end melalui `src/inference.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_dropout_data, load_dropout_reasons_data
from src.inference import VocationalInferenceEngine

df = load_dropout_data()
df_reasons = load_dropout_reasons_data()
print(f"Dataset peserta lokal: {len(df)} baris | Kasus Drop-Out: {df['status_dropout'].sum()} ({df['status_dropout'].mean()*100:.2f}%)")
print(df[['student_id', 'jenis_program', 'zona', 'jarak_km', 'kehadiran_persen', 'status_dropout']].head(5).to_string(index=False))

Dataset peserta lokal: 349 baris | Kasus Drop-Out: 14 (4.01%)
student_id jenis_program   zona  jarak_km  kehadiran_persen  status_dropout
   STU-001       REGULER Zone A      12.0              96.2               0
   STU-002       REGULER Zone A       6.0              94.1               0
   STU-003       REGULER Zone A       3.0              91.9               0
   STU-004       REGULER Zone A       7.0              70.8               1
   STU-005       REGULER Zone A      12.0              95.3               0


## 1. Analisis Diagnostik: Kesenjangan Kehadiran, Program, & Alasan Keluar

In [2]:
print("=== Rata-Rata Kehadiran (%) Selesai vs Drop-Out ===")
print(df.groupby('status_dropout')['kehadiran_persen'].agg(['count', 'mean', 'median']).round(2).to_string())

print("\n=== Tingkat Drop-Out per Program ===")
prog_summary = (
    df.groupby('jenis_program')
      .agg(total=('student_id', 'count'), dropout=('status_dropout', 'sum'), rate_pct=('status_dropout', lambda s: round(s.mean()*100, 2)))
      .sort_values('rate_pct', ascending=False)
)
print(prog_summary.to_string())

print("\n=== Pareto Alasan Keluar (Kualitatif) ===")
print(df_reasons['alasan_keluar'].value_counts().to_string())

=== Rata-Rata Kehadiran (%) Selesai vs Drop-Out ===
                count   mean  median
status_dropout                      
0                 335  95.74  100.00
1                  14  57.67   53.42

=== Tingkat Drop-Out per Program ===
               total  dropout  rate_pct
jenis_program                          
REGULER           89        6      6.74
PKW               45        3      6.67
HARIAN            37        2      5.41
PKK              178        3      1.69

=== Pareto Alasan Keluar (Kualitatif) ===
alasan_keluar
Bekerja                3
Kesibukan              3
Menikah                2
Ekonomi                2
Sakit                  2
Modul terlalu susah    2


## 2. Replikasi Evaluasi Dual-Stage Pipeline (Lokal)

In [3]:
edu_map = {'SD/Sederajat': 0, 'SMP/Sederajat': 1, 'SMA/SMK/Sederajat': 2, 'Perguruan Tinggi': 3}
jk_map = {'P': 0, 'L': 1}
zones = ['Zone A', 'Zone B', 'Zone C', 'Zone D', 'Zone E', 'Zone F', 'Other Zones']
progs = ['PKK', 'PKW', 'REGULER', 'HARIAN']

X_base = pd.DataFrame({
    'paket_jp_diambil': df['paket_jp_diambil'],
    'bulan_mulai': df['bulan_mulai'],
    'jenis_kelamin_encoded': df['jenis_kelamin'].map(jk_map),
    'usia_masuk': df['usia_masuk'],
    'pendidikan_encoded': df['pendidikan'].map(edu_map),
    'jarak_km': df['jarak_km'],
})
for p in progs:
    X_base[f'program_{p}'] = (df['jenis_program'] == p).astype(int)
for z in zones:
    X_base[f'zona_{z}'] = (df['zona'] == z).astype(int)

X_stage2 = X_base.copy()
X_stage2['kehadiran_persen'] = df['kehadiran_persen']
y = df['status_dropout']

idx_tr, idx_te = train_test_split(df.index, test_size=0.2, random_state=42, stratify=y)

# Stage 1: Logistic Regression (thr=0.25)
pipe_s1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
pipe_s1.fit(X_base.loc[idx_tr], y.loc[idx_tr])
prob_s1 = pipe_s1.predict_proba(X_base.loc[idx_te])[:, 1]
pred_s1 = (prob_s1 >= 0.25).astype(int)

# Stage 2: Gradient Boosting (thr=0.50)
pipe_s2 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', GradientBoostingClassifier(learning_rate=0.08, n_estimators=100, random_state=42))
])
pipe_s2.fit(X_stage2.loc[idx_tr], y.loc[idx_tr])
prob_s2 = pipe_s2.predict_proba(X_stage2.loc[idx_te])[:, 1]
pred_s2 = (prob_s2 >= 0.50).astype(int)

print(f"Stage 1 (Pre-Enrollment, thr=0.25) -> Recall: {recall_score(y.loc[idx_te], pred_s1):.3f} | ROC-AUC: {roc_auc_score(y.loc[idx_te], prob_s1):.3f}")
print(f"Stage 2 (Ongoing, thr=0.50)        -> Precision: {precision_score(y.loc[idx_te], pred_s2, zero_division=0):.3f} | Recall: {recall_score(y.loc[idx_te], pred_s2):.3f} | F1: {f1_score(y.loc[idx_te], pred_s2):.3f} | ROC-AUC: {roc_auc_score(y.loc[idx_te], prob_s2):.3f}")

Stage 1 (Pre-Enrollment, thr=0.25) -> Recall: 0.667 | ROC-AUC: 0.741
Stage 2 (Ongoing, thr=0.50)        -> Precision: 1.000 | Recall: 0.667 | F1: 0.800 | ROC-AUC: 0.955


## 3. Uji Inferensi Artefak Model Ter-deploy (`models/*.pkl`)

In [4]:
engine = VocationalInferenceEngine()
res = engine.predict_dropout(
    jenis_program='REGULER',
    bulan_mulai=3,
    jenis_kelamin='P',
    usia_masuk=26.0,
    pendidikan='SMA/SMK/Sederajat',
    jarak_km=7.0,
    zona='Zone A',
    kehadiran_persen=62.0
)
for k, v in res.items():
    print(f"{k}: {v}")

stage1_pre_enrollment: {'probability_dropout': 0.6802, 'threshold': 0.25, 'predicted_code': 1, 'predicted_label': 'KELUAR (Berisiko Drop-Out)'}
stage2_ongoing_monitoring: {'probability_dropout': 0.9997, 'threshold': 0.5, 'predicted_code': 1, 'predicted_label': 'KELUAR (Berisiko Drop-Out)'}
recommendation: Intervensi segera: lakukan konseling personal, evaluasi kendala kehadiran/transportasi, dan tawarkan fleksibilitas jadwal praktik.
